# 03. Normalization

### Objective
Derive rate features and apply MinMaxScaler to all telemetry features.

### I/O
- **Reads**: `../../data/processed/2_cleaned_telemetry_for_modelling.csv`
- **Writes**: `../../data/processed/3_normalized_telemetry.csv`
- **Writes**: `../../data/models/scaler_params.json`

In [1]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import json, os

INPUT_PATH  = '../../data/processed/2_cleaned_telemetry_for_modelling.csv'
OUTPUT_PATH = '../../data/processed/3_normalized_telemetry.csv'
MODEL_DIR   = '../../data/models'
os.makedirs(MODEL_DIR, exist_ok=True)

print('Libraries imported')

Libraries imported


In [2]:
df = pd.read_csv(INPUT_PATH)
print(f'Loaded {len(df)} rows')

Loaded 3240 rows


In [3]:
# Outlier adjustment before feature derivation and scaling.
# Time columns are bounded by the 30s telemetry window - anything above 30
# is a measurement artefact (float jitter or tracker bug).
# Count columns have no hard physical cap, but extreme tails compress the scaler.
# Cap at p99 so the top 1% is adjusted to the 99th-percentile value, not removed.
# distanceTraveled is skipped - its distribution has no IQR outliers.

TIME_COLS  = ['timeInCombat', 'timeOutOfCombat', 'timeSprinting', 'timeNearInteractables']
COUNT_COLS = ['enemiesHit', 'damageDone', 'kills', 'itemsCollected', 'pickupAttempts']

print('--- Time column clamping (hard cap = 30s) ---')
for col in TIME_COLS:
    if col in df.columns:
        n = (df[col] > 30).sum()
        df[col] = df[col].clip(upper=30)
        print(f'  {col}: {n} rows clamped to 30s')

print()
print('--- Count column clamping (p99 cap) ---')
for col in COUNT_COLS:
    if col in df.columns:
        cap = df[col].quantile(0.99)
        n = (df[col] > cap).sum()
        df[col] = df[col].clip(upper=cap)
        print(f'  {col}: {n} rows clamped to p99={cap:.2f}')

print()
print(f'Outlier adjustment done. Shape unchanged: {df.shape}')

--- Time column clamping (hard cap = 30s) ---
  timeInCombat: 1 rows clamped to 30s
  timeOutOfCombat: 247 rows clamped to 30s
  timeSprinting: 5 rows clamped to 30s
  timeNearInteractables: 17 rows clamped to 30s

--- Count column clamping (p99 cap) ---
  enemiesHit: 32 rows clamped to p99=23.00
  damageDone: 26 rows clamped to p99=298.67
  kills: 23 rows clamped to p99=3.00
  itemsCollected: 19 rows clamped to p99=8.00
  pickupAttempts: 31 rows clamped to p99=25.00

Outlier adjustment done. Shape unchanged: (3240, 36)


In [4]:
# v2.2 derived features - computed here before scaling so they get normalized
# along with everything else.
#
# damage_per_hit: a sniper or heavy-weapon player hits few enemies but deals
# massive damage each hit. Raw enemiesHit alone underrepresents those players,
# so I divide damageDone by hit count to get weapon-class-agnostic combat intensity.
#
# pickup_attempt_rate: true collectors walk up to items and actively attempt pickups.
# Explorers pass near items incidentally but rarely attempt them. Dividing pickupAttempts
# by time near interactables captures that deliberate collection intent.
df['damage_per_hit']       = df['damageDone']      / df['enemiesHit'].clip(lower=1)
df['pickup_attempt_rate']  = df['pickupAttempts']  / df['timeNearInteractables'].clip(lower=1)

print('Derived features added:')
print(f'  damage_per_hit range: [{df["damage_per_hit"].min():.3f}, {df["damage_per_hit"].max():.3f}]')
print(f'  pickup_attempt_rate range: [{df["pickup_attempt_rate"].min():.3f}, {df["pickup_attempt_rate"].max():.3f}]')

Derived features added:
  damage_per_hit range: [0.000, 18.667]
  pickup_attempt_rate range: [0.000, 25.000]


In [5]:
features_to_normalize = [
    'enemiesHit', 'damageDone', 'timeInCombat', 'kills',
    'itemsCollected', 'pickupAttempts', 'timeNearInteractables',
    'distanceTraveled', 'timeSprinting', 'timeOutOfCombat',
    'damage_per_hit',
    'pickup_attempt_rate',
]

available_features = [f for f in features_to_normalize if f in df.columns]
print(f'Features to normalize: {len(available_features)}')

Features to normalize: 12


Experiment B was run to understand which normalization produces the best clustering separation.
Experiment B is at: `experiments/experiment_B_clustering_config/Experiment_B_Complete.ipynb`

The grid search tested minmax_uniform, minmax_log_sparse (log1p on skewed combat features first),
and RobustScaler across 108 configurations on raw features. The actual top result from that
experiment was **minmax_log_sparse** (K=3, silhouette=0.3928) - that finding applies to
raw-feature clustering and is documented there.

This notebook normalizes features for a different purpose: computing activity score means
(score_combat, score_collect, score_explore) in notebook 04, which feed into pct percentages
that notebook 05 clusters on. For that use case, plain MinMaxScaler is sufficient - features
just need to be on the same [0,1] scale so the means are comparable across archetypes.
The log-transform advantage only applies when doing Euclidean distance directly in raw feature space.

Because of this, uniform MinMaxScaler is applied below.

In [6]:
scaler = MinMaxScaler()
df[available_features] = scaler.fit_transform(df[available_features].fillna(0))

print('Applied uniform MinMaxScaler [0, 1]')
print(df[available_features].describe())

Applied uniform MinMaxScaler [0, 1]
        enemiesHit   damageDone  timeInCombat        kills  itemsCollected  \
count  3240.000000  3240.000000   3240.000000  3240.000000     3240.000000   
mean      0.131817     0.156944      0.181986     0.140741        0.151620   
std       0.197295     0.218056      0.226810     0.226567        0.205933   
min       0.000000     0.000000      0.000000     0.000000        0.000000   
25%       0.000000     0.000000      0.000000     0.000000        0.000000   
50%       0.000000     0.000000      0.066667     0.000000        0.125000   
75%       0.217391     0.312500      0.333333     0.333333        0.250000   
max       1.000000     1.000000      1.000000     1.000000        1.000000   

       pickupAttempts  timeNearInteractables  distanceTraveled  timeSprinting  \
count     3240.000000            3240.000000       3240.000000    3240.000000   
mean         0.145963               0.125844          0.455634       0.177402   
std          0.208

In [7]:
df.to_csv(OUTPUT_PATH, index=False)
print(f'Saved to {OUTPUT_PATH}')

Saved to ../../data/processed/3_normalized_telemetry.csv


In [8]:
# Export scaler parameters so the demo frontend can normalize live player input
# using the exact same min/max values this training data was scaled with.
scaler_params = {
    'features':    list(available_features),
    'data_min':    scaler.data_min_.tolist(),
    'data_max':    scaler.data_max_.tolist(),
    'data_range':  (scaler.data_max_ - scaler.data_min_).tolist(),
    'min_value':   float(scaler.feature_range[0]),
    'max_value':   float(scaler.feature_range[1])
}

SCALER_OUT = os.path.join(MODEL_DIR, 'scaler_params.json')
with open(SCALER_OUT, 'w') as f:
    json.dump(scaler_params, f, indent=2)

print(f'Scaler parameters exported to {SCALER_OUT}')
print(f'Features included ({len(available_features)}): {available_features}')

Scaler parameters exported to ../../data/models\scaler_params.json
Features included (12): ['enemiesHit', 'damageDone', 'timeInCombat', 'kills', 'itemsCollected', 'pickupAttempts', 'timeNearInteractables', 'distanceTraveled', 'timeSprinting', 'timeOutOfCombat', 'damage_per_hit', 'pickup_attempt_rate']
